# Laborator 3 — Explorarea și preprocesarea datelor (varianta cu soluții)

**Curs:** Machine Learning in Python · Laborator asociat Cursului 3 (Preprocesarea datelor)

### Ce vei învăța în acest laborator
- Explorarea unui set de date real (consum de combustibil și emisii CO₂): statistici, distribuții, corelații
- Interpretarea statisticilor descriptive (omogenitate, outlieri, asimetrie, boltire)
- Transformarea logaritmică a unei variabile asimetrice
- Detectarea și tratarea outlierilor (regula IQR)
- Codificarea variabilelor categoriale și scalarea celor numerice
- Gestionarea valorilor lipsă: eliminare, imputare simplă și avansată (set de date Boston Housing)

> 💡 Rulează celulele în ordine, de sus în jos. Celulele marcate **📝 Exercițiu** sunt pentru tine.

## 0. Importul librăriilor

In [ ]:
# Librării de bază
import os
import numpy as np
import pandas as pd

# Vizualizare
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesare (scikit-learn)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

# Partea 2 — Valori lipsă: setul de date Boston Housing

**Despre setul de date.** Datele descriu diferite zone din Boston și au scopul de a prezice prețul median al locuințelor dintr-o zonă (`MEDV`). Observațiile sunt zone/orașe, nu locuințe individuale:

- `CRIM` — rata criminalității pe cap de locuitor
- `ZN` — proporția de teren rezidențial pentru loturi de peste 25.000 sq.ft.
- `INDUS` — proporția de hectare comerciale non-retail pe oraș
- `CHAS` — variabilă dummy Charles River (1 dacă zona delimitează râul; 0 în caz contrar)
- `NOX` — concentrația de oxizi de azot
- `RM` — numărul mediu de camere pe locuință
- `AGE` — proporția unităților construite înainte de 1940
- `DIS` — distanța ponderată până la cinci centre de angajare
- `RAD` — indice de accesibilitate la autostrăzi
- `TAX` — cota de impozit pe proprietate
- `PTRATIO` — raportul elevi/profesori
- `LSTAT` — procentul populației cu statut socio-economic scăzut
- `MEDV` — prețul median al locuințelor (în mii $)

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/real_estate_data.csv"
LOCAL = "real_estate_data.csv"

try:
    df = pd.read_csv(URL)
    df.to_csv(LOCAL, index=False)          # salvăm o copie locală
except Exception:
    print("Descărcarea a eșuat — folosesc copia locală.")
    df = pd.read_csv(LOCAL)

df_raw = df.copy()      # păstrăm datele originale, pentru comparații ulterioare
df.head()

In [ ]:
df.info()

In [ ]:
# Procentul de valori lipsă pe fiecare coloană
(df.isna().mean() * 100).round(2).sort_values(ascending=False)

## 8. Curățarea bazei de date — eliminarea coloanelor cu multe valori lipsă

Definim o funcție care elimină coloanele ce depășesc un **prag procentual** de valori lipsă.

In [ ]:
def drop_columns_with_high_nulls(df, threshold_percentage):
    """Elimină coloanele cu mai mult de `threshold_percentage` % valori lipsă."""
    threshold = len(df) * (threshold_percentage / 100)
    dropped_columns = df.columns[df.isnull().sum() > threshold]
    df_cleaned = df.drop(columns=dropped_columns)
    return df_cleaned, dropped_columns

for prag in [10, 3]:
    df_cleaned, dropped = drop_columns_with_high_nulls(df, prag)
    print(f"Prag {prag:>2}%  ->  coloane eliminate: {list(dropped)}")

### 🔎 Interpretare
- La un prag de **10%** nu este eliminată nicio coloană (fiecare coloană afectată are ~4% valori lipsă).
- La un prag de **3%** sunt eliminate toate cele 6 coloane afectate — am pierde aproape jumătate din variabile.

Alegerea pragului este o decizie a analistului: aici, **imputarea** este mai potrivită decât eliminarea, deoarece pierderea de informație ar fi mare.

## 9. Imputarea valorilor lipsă cu media

In [ ]:
num_data = df.select_dtypes(include=['int64', 'float64']).columns

df_mean = df.copy()
for col in num_data:
    df_mean[col] = df_mean[col].fillna(df_mean[col].mean())

print("Valori lipsă rămase:", df_mean.isna().sum().sum())

## 📝 Exerciții

Lucrați individual; folosiți `df_raw` (datele originale, cu valori lipsă) și `df_mean` (imputarea cu media).

**Exercițiul 1.** Analizați baza de date după imputarea cu media: comparați, pentru variabila `LSTAT`, media și deviația standard **înainte** și **după** imputare. Ce observați la deviația standard?

In [ ]:
# SOLUȚIE
comp = pd.DataFrame({'înainte': df_raw['LSTAT'].describe(), 'după (medie)': df_mean['LSTAT'].describe()})
comp.round(3)

> **Interpretare:** media rămâne aproape neschimbată, dar **deviația standard scade** — imputarea cu media „înghesuie” valorile în jurul mediei și subestimează variabilitatea reală a datelor.

**Exercițiul 2.** Imputați valorile lipsă și cu **mediana** (`SimpleImputer(strategy='median')`) și cu **KNN** (`KNNImputer(n_neighbors=5)`). Salvați rezultatele în `df_median` și `df_knn`.

In [ ]:
# SOLUȚIE
df_median = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(df_raw), columns=df_raw.columns)
df_knn = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(df_raw), columns=df_raw.columns)

print("Lipsuri median:", df_median.isna().sum().sum(), "| Lipsuri KNN:", df_knn.isna().sum().sum())

**Exercițiul 3.** Comparați cele trei metode de imputare pe variabila `LSTAT`: tabel cu media și deviația standard + histogramele suprapuse. Care metodă păstrează cel mai bine forma distribuției originale?

In [ ]:
# SOLUȚIE
tabel = pd.DataFrame({
    'original (fără NaN)': df_raw['LSTAT'].dropna().agg(['mean', 'std']),
    'medie': df_mean['LSTAT'].agg(['mean', 'std']),
    'mediană': df_median['LSTAT'].agg(['mean', 'std']),
    'KNN': df_knn['LSTAT'].agg(['mean', 'std']),
}).round(3)
display(tabel)

plt.figure(figsize=(9, 4))
sns.kdeplot(df_raw['LSTAT'].dropna(), label='original', linewidth=2.5, color='black')
sns.kdeplot(df_mean['LSTAT'], label='medie')
sns.kdeplot(df_median['LSTAT'], label='mediană')
sns.kdeplot(df_knn['LSTAT'], label='KNN')
plt.legend(); plt.title('LSTAT — comparație metode de imputare'); plt.show()

> **Interpretare:** imputarea cu media/mediana creează un „vârf” artificial în jurul valorii centrale; KNN folosește informația din celelalte variabile și păstrează mai bine forma distribuției.

**Exercițiul 4 (opțional).** Creați, înainte de imputare, o variabilă binară `LSTAT_missing` (1 dacă valoarea lipsea, 0 altfel). Câte observații au avut valori lipsă la `LSTAT`?

In [ ]:
# SOLUȚIE
df_flag = df_mean.copy()
df_flag['LSTAT_missing'] = df_raw['LSTAT'].isna().astype(int)
print("Observații cu LSTAT lipsă:", df_flag['LSTAT_missing'].sum())

**Exercițiul 5 (opțional).** Detectați outlierii variabilei `CRIM` cu regula IQR (folosiți `limite_iqr`) și aplicați limitarea (clip). Câți outlieri existau?

In [ ]:
# SOLUȚIE
jos, sus = limite_iqr(df_mean['CRIM'])
n_out = ((df_mean['CRIM'] < jos) | (df_mean['CRIM'] > sus)).sum()
print(f"Outlieri CRIM: {n_out}")
df_mean['CRIM_capped'] = df_mean['CRIM'].clip(lower=jos, upper=sus)
df_mean[['CRIM', 'CRIM_capped']].describe().round(3)

## Recapitulare

În acest laborator ai exersat:
- ✅ interpretarea statisticilor descriptive (omogenitate, outlieri, asimetrie, boltire)
- ✅ transformarea logaritmică și compararea distribuțiilor înainte/după
- ✅ detectarea și limitarea outlierilor (regula IQR)
- ✅ One-Hot Encoding pentru variabile categoriale și scalarea variabilelor numerice
- ✅ tratarea valorilor lipsă: eliminare pe bază de prag, imputare cu medie / mediană / KNN

➡️ **Următorul laborator** introduce primele modele de **regresie** (Cursul 4), pornind de la datele pregătite aici.